In [1]:

from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

d:\LangGraph\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model =  ChatGroq(
        model = "llama-3.1-8b-instant"
    )

In [4]:
MAX_TOKENS = 150

In [5]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",   # last messages rkhne hain                     
        token_counter=count_tokens_approximately, # token counting function
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [7]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [8]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Amna."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 9
Hi, my name is Amna.


'Nice to meet you, Amna. Is there something I can help you with or would you like to chat?'

In [11]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Amna."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 140
Hi, my name is Amna.
Nice to meet you, Amna. Is there something I can help you with or would you like to chat?
Hi, my name is Nitish.
Nice to meet you, Nitish. It seems we had a brief introduction earlier, Amna was the one who started the conversation. How are you doing today?
Hi, my name is Amna.
Hello again Amna. I think Nitish and I were just chatting. Do you want to join the conversation or start a new one?
Hi, my name is Amna.


'It seems like Amna is back. Don\'t worry, I won\'t keep track of multiple people in this conversation. Each time you say "Hi, my name is Amna", it\'s a new message. If you\'d like to continue the conversation with Nitish or start a new one, just let me know.'